# Level 5 Term 1 (continued): Teddy Bear Chase
## Sessions 4-7 - DRAFT for teacher review

**Project goal:** the UGOT uses YOLO to find a toy teddy bear, shows its box, center x (`cx`) and width (`w`), turns to face it, and drives up to it.

**Rules for this unit**
- No PID / Kp. The robot is controlled with simple `if / elif / else` rules and fixed speeds.
- The UGOT camera is only 640 x 480, so we also learn how to get better detection from the same camera.

**Assumes** the setup cells from Sessions 1-3 have been run (`got`, `model`, `np`, `cv2`, `time`, `display`, `Image`). The cell below repeats them in case you start fresh.

In [ ]:
# setup (skip if already run)
import time
import numpy as np
import cv2
from IPython.display import display, Image
from ugot import ugot
from ultralytics import YOLO

got = ugot.UGOT()
got.initialize('192.168.1.230')   # change to your robot's IP
model = YOLO("yolo26n.pt")
got.open_camera()
print("Ready.")

In [ ]:
# helper: get one picture from the UGOT camera as a 3D array (BGR)
def get_frame():
    picture_raw = got.read_camera_data()
    if picture_raw is None:
        return None
    return cv2.imdecode(np.frombuffer(picture_raw,np.uint8),cv2.IMREAD_COLOR)

frame = get_frame()
print(f"Shape: {frame.shape}   (height, width, colors)")

# Session 4: Find the teddy and read its numbers
1. Which name does YOLO use for a teddy bear?
2. Keep only the teddy bear
3. Center x and width: `xywh` vs `xyxy`
4. `find_teddy()` function
5. Live video with the teddy box
6. Test table and baseline

## 1. What does YOLO call a teddy bear?

In [ ]:
# print all the names YOLO knows
for number,name in model.names.items():
    print(number,name)

In [ ]:
# search the names
for number,name in model.names.items():
    if "teddy" in name:
        print(f"Found it! Number {number} is {name}.")

## 2. Keep only the teddy bear
Put the teddy in front of the robot, then run.

In [ ]:
# look at everything YOLO sees
frame = get_frame()
results = model(frame,verbose = False)

for box in results[0].boxes:
    class_id = int(box.cls[0])
    name = model.names[class_id]
    conf = float(box.conf[0])
    print(f"I see {name} with the confidence of {conf:.2f}.")

In [ ]:
# keep only the teddy bear
frame = get_frame()
results = model(frame,verbose = False)

teddy_found = False
for box in results[0].boxes:
    class_id = int(box.cls[0])
    name = model.names[class_id]
    conf = float(box.conf[0])
    if name == "teddy bear":
        teddy_found = True
        print(f"Teddy found! confidence = {conf:.2f}")

if not teddy_found:
    print("No teddy in this picture.")

## 3. Center x and width
`box.xyxy` gives the corners: `x1, y1, x2, y2`.
`box.xywh` gives the **center x, center y, width, height**.

Two ways to get the same numbers. Do they match?

In [ ]:
frame = get_frame()
results = model(frame,verbose = False)

for box in results[0].boxes:
    name = model.names[int(box.cls[0])]
    if name == "teddy bear":
        # way 1: from the corners
        x1,y1,x2,y2 = map(int,box.xyxy[0])
        cx_1 = (x1 + x2)//2
        w_1 = x2 - x1

        # way 2: xywh gives them directly
        cx_2,cy_2,w_2,h_2 = map(int,box.xywh[0])

        print(f"Way 1: cx = {cx_1}, w = {w_1}")
        print(f"Way 2: cx = {cx_2}, w = {w_2}")

## 4. Make a function: `find_teddy()`
Input: one picture. Output: `(cx, w, conf)` of the most confident teddy, or `None` if there is no teddy.

In [ ]:
def find_teddy(frame):
    results = model(frame,verbose = False)

    best = None
    best_conf = 0
    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        if name == "teddy bear" and conf > best_conf:
            cx,cy,w,h = map(int,box.xywh[0])
            best = (cx,w,conf)
            best_conf = conf
    return best

# test
frame = get_frame()
print(find_teddy(frame))

## 5. Live video with the teddy box
Stop with the **stop button** of the notebook (Interrupt the kernel).

In [ ]:
# live video: box, center dot, cx, w
frame = get_frame()
ok,picture_jpg = cv2.imencode(".jpg",frame)
handle = display(Image(picture_jpg.tobytes()),display_id = True)

while True:
    frame = get_frame()
    if frame is None:
        continue

    results = model(frame,verbose = False)
    for box in results[0].boxes:
        name = model.names[int(box.cls[0])]
        conf = float(box.conf[0])
        if name == "teddy bear":
            x1,y1,x2,y2 = map(int,box.xyxy[0])
            cx = (x1 + x2)//2
            cy = (y1 + y2)//2
            w = x2 - x1
            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,0),2)
            cv2.circle(frame,(cx,cy),6,(255,0,0),-1)
            cv2.putText(frame,f"teddy {conf:.2f}",(x1,y1-30),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)
            cv2.putText(frame,f"cx:{cx} w:{w}",(x1,y1-10),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

    ok,picture_jpg = cv2.imencode(".jpg",frame)
    if ok:
        handle.update(Image(picture_jpg.tobytes()))
    time.sleep(0.05)

## 6. Test table and baseline
Put the teddy at **left / center / right** and at **30 cm / 60 cm / 90 cm**. Run the cell once for each place, then fill in the table.

| Position | Distance | Found? | cx | w | conf |
|---|---|---|---|---|---|
| left | 30 cm | | | | |
| center | 30 cm | | | | |
| right | 30 cm | | | | |
| left | 60 cm | | | | |
| ... | | | | | |

In [ ]:
frame = get_frame()
result = find_teddy(frame)
if result is None:
    print("Found? NO")
else:
    cx,w,conf = result
    print(f"Found? YES   cx = {cx}   w = {w}   conf = {conf:.2f}")

In [ ]:
# baseline: take 20 pictures, count how often the teddy is found
found = 0
total = 20
for i in range(total):
    frame = get_frame()
    if find_teddy(frame) is not None:
        found += 1
    time.sleep(0.2)
print(f"Baseline: found the teddy in {found} of {total} pictures.")

**Write down your baseline number.** In Session 5 we try to beat it.

**Think:** What happens when there is no teddy? What if there are two teddies?

# Session 5: Make the detector more reliable
The UGOT camera is small (640 x 480). Making the picture bigger does **not** add detail, so we improve the **scene**, the **settings** and the **way we use the results**.

1. Better scene (no code)
2. Build a small test set
3. Confidence threshold
4. Bigger model
5. Brighten the picture
6. Vote across frames
7. Compare and chart

## 1. Better scene checklist
Try each one and watch the live video from Session 4.
- [ ] Light the teddy from the **front** (no bright window behind it)
- [ ] Plain background that is a **different color** from the teddy
- [ ] Clean the camera lens
- [ ] Keep the teddy **20-100 cm** away
- [ ] Teddy sitting upright, not hidden by other objects

## 2. Build a mini test set
- 20 photos **with** the teddy (move it around)
- 10 photos **without** the teddy

We save them as files so we can test every setting on the **same** pictures.

In [ ]:
import os
os.makedirs("teddy_test/with_teddy",exist_ok=True)
os.makedirs("teddy_test/no_teddy",exist_ok=True)

def collect_photos(folder,total,gap=1.5):
    for i in range(total):
        print(f"Photo {i+1}/{total} - change the teddy position now...")
        time.sleep(gap)
        frame = get_frame()
        cv2.imwrite(f"{folder}/photo_{i+1:02d}.jpg",frame)
    print("Done.")

collect_photos("teddy_test/with_teddy",20)

In [ ]:
# now remove the teddy from the camera view
collect_photos("teddy_test/no_teddy",10)

In [ ]:
import glob
with_files = sorted(glob.glob("teddy_test/with_teddy/*.jpg"))
no_files = sorted(glob.glob("teddy_test/no_teddy/*.jpg"))
print(len(with_files),"photos with teddy,",len(no_files),"photos without teddy")

# scoring function: hits = teddy found in "with" photos; false alarms = teddy found in "no" photos
def score(detect_function):
    hits = 0
    for file in with_files:
        if detect_function(cv2.imread(file)):
            hits += 1
    false_alarms = 0
    for file in no_files:
        if detect_function(cv2.imread(file)):
            false_alarms += 1
    return hits,false_alarms

# the baseline detector from Session 4
def detect_baseline(frame):
    return find_teddy(frame) is not None

hits,false_alarms = score(detect_baseline)
print(f"Baseline: hits {hits}/{len(with_files)}, false alarms {false_alarms}/{len(no_files)}")

## 3. Confidence threshold
`conf=0.25` means YOLO keeps guesses it is at least 25% sure about. A low number finds more teddies, but also more false alarms.

In [ ]:
def make_detector(yolo_model,conf_value,brighten = False):
    def detect(frame):
        if brighten:
            frame = cv2.convertScaleAbs(frame,alpha = 1.3,beta = 20)
        results = yolo_model(frame,conf = conf_value,verbose = False)
        for box in results[0].boxes:
            if yolo_model.names[int(box.cls[0])] == "teddy bear":
                return True
        return False
    return detect

for conf_value in [0.7,0.5,0.4,0.3,0.25]:
    hits,false_alarms = score(make_detector(model,conf_value))
    print(f"conf = {conf_value}: hits {hits}/{len(with_files)}, false alarms {false_alarms}/{len(no_files)}")

## 4. A bigger model
`yolo26n` is the smallest (fastest). `yolo26s` is bigger: usually more accurate, but slower. The model runs on the laptop, not on the robot.

In [ ]:
model_s = YOLO("yolo26s.pt")

start = time.time()
hits,false_alarms = score(make_detector(model_s,0.4))
seconds = time.time() - start
print(f"yolo26s: hits {hits}/{len(with_files)}, false alarms {false_alarms}/{len(no_files)}, time {seconds:.1f} s")

start = time.time()
hits,false_alarms = score(make_detector(model,0.4))
seconds = time.time() - start
print(f"yolo26n: hits {hits}/{len(with_files)}, false alarms {false_alarms}/{len(no_files)}, time {seconds:.1f} s")

## 5. Brighten the picture
`cv2.convertScaleAbs(frame, alpha, beta)` : `alpha` makes the contrast bigger, `beta` makes it brighter. This sometimes helps and sometimes does not. **Measure it.**

In [ ]:
frame = cv2.imread(with_files[0])
bright = cv2.convertScaleAbs(frame,alpha = 1.3,beta = 20)
both = np.hstack((frame,bright))
ok,picture_jpg = cv2.imencode(".jpg",both)
display(Image(picture_jpg.tobytes()))

In [ ]:
for brighten in [False,True]:
    hits,false_alarms = score(make_detector(model,0.4,brighten))
    print(f"brighten = {brighten}: hits {hits}/{len(with_files)}, false alarms {false_alarms}/{len(no_files)}")

## 6. Vote across frames
One frame can be wrong. We keep the last 5 results in a list and trust the teddy only if it was found in **at least 3 of them**.

In [ ]:
recent = []   # the last 5 results: True or False

def teddy_voted(found_now):
    recent.append(found_now)
    if len(recent) > 5:
        recent.pop(0)        # remove the oldest result
    return recent.count(True) >= 3

# test with live pictures
for i in range(15):
    frame = get_frame()
    found_now = find_teddy(frame) is not None
    trusted = teddy_voted(found_now)
    print(f"now: {found_now}   recent: {recent}   trusted: {trusted}")
    time.sleep(0.2)

## 7. Compare and chart

In [ ]:
import matplotlib.pyplot as plt

settings = {
    "baseline":  make_detector(model,0.5),
    "conf 0.3":  make_detector(model,0.3),
    "bright":    make_detector(model,0.4,True),
    "model s":   make_detector(model_s,0.4),
}

names = []
hits_list = []
for name,detector in settings.items():
    hits,false_alarms = score(detector)
    names.append(name)
    hits_list.append(hits)
    print(f"{name}: hits {hits}, false alarms {false_alarms}")

plt.bar(names,hits_list,color = "orange")
plt.ylabel(f"hits out of {len(with_files)}")
plt.title("Which setting finds the teddy best?")
plt.show()

**Write down your best setting.** We will use it in Sessions 6 and 7.

In [ ]:
# the settings we will use from now on (change after your experiment)
BEST_MODEL = model
BEST_CONF = 0.4
BRIGHTEN = False

def find_teddy_best(frame):
    # same as find_teddy, but with our best settings
    if BRIGHTEN:
        frame = cv2.convertScaleAbs(frame,alpha = 1.3,beta = 20)
    results = BEST_MODEL(frame,conf = BEST_CONF,verbose = False)

    best = None
    best_conf = 0
    for box in results[0].boxes:
        name = BEST_MODEL.names[int(box.cls[0])]
        conf = float(box.conf[0])
        if name == "teddy bear" and conf > best_conf:
            cx,cy,w,h = map(int,box.xywh[0])
            best = (cx,w,conf)
            best_conf = conf
    return best

# Session 6: Turn to face the teddy
1. Which way does the robot turn?
2. Rules: left / right / centered
3. Stop - look - go (clear pictures)
4. If the teddy is lost
5. Face the teddy program

## 1. Which way does the robot turn?
The robot turns for 1 second. Write down: does `z = 15` turn **left** or **right**?

In [ ]:
got.mecanum_move_xyz(0,0,15)
time.sleep(1)
got.mecanum_stop()

In [ ]:
# Teacher/students: if z = +15 turns the robot to the RIGHT, TURN_SIGN = 1.
# If it turns to the LEFT, change TURN_SIGN to -1.
TURN_SIGN = 1

## 2. The rules
The camera picture is 640 pixels wide, so the middle is `320`.

```
error_x = cx - 320
error_x is negative  -> teddy is on the LEFT
error_x is positive  -> teddy is on the RIGHT
error_x is small     -> CENTERED (inside the dead zone)
```

In [ ]:
FRAME_CENTER = 320
DEAD_ZONE = 40       # how many pixels counts as "centered"
TURN_SPEED = 15

def decide_turn(cx):
    error_x = cx - FRAME_CENTER
    if error_x < -DEAD_ZONE:
        return "left"
    elif error_x > DEAD_ZONE:
        return "right"
    else:
        return "centered"

# test the rules without moving the robot
for test_cx in [50,250,320,380,600]:
    print(test_cx,decide_turn(test_cx))

## 3. Stop - look - go
A moving robot makes **blurry** pictures and the picture can be **old**. So we:
1. move a **short** step,
2. **stop**,
3. wait a moment,
4. take a **fresh** picture and look.

In [ ]:
def fresh_frame(): 
    got.mecanum_stop()
    time.sleep(0.3)          # wait until the robot stops shaking
    get_frame()              # throw away one old picture
    return get_frame()       # use a fresh one

def turn_step(direction):
    # direction is "left" or "right"; one short turning step
    if direction == "right":
        z = TURN_SPEED * TURN_SIGN
    else:
        z = -TURN_SPEED * TURN_SIGN
    got.mecanum_move_xyz(0,0,z)
    time.sleep(0.2)
    got.mecanum_stop()

## 4. What if the teddy is lost?
We count the misses. After 5 misses in a row, the robot spins slowly to search. After 40 steps in total, it gives up.

## 5. The face-the-teddy program

In [ ]:
def face_teddy(max_steps = 40):
    missed = 0
    for step in range(max_steps):
        frame = fresh_frame()
        result = find_teddy_best(frame)

        if result is None:
            missed += 1
            print(f"Step {step}: no teddy ({missed} missed)")
            if missed >= 5:
                turn_step("right")      # search: turn a little
            continue

        missed = 0
        cx,w,conf = result
        direction = decide_turn(cx)
        print(f"Step {step}: cx = {cx}  w = {w}  -> {direction}")

        if direction == "centered":
            got.show_light_rgb_effect(0,255,0,3)
            got.mecanum_stop()
            return True
        turn_step(direction)

    got.mecanum_stop()
    return False

if face_teddy():
    got.play_audio_tts("I see the teddy!")
else:
    print("I could not find the teddy.")
got.turn_off_lights()

**Practice**
1. Change `DEAD_ZONE` to 10, then 100. What happens?
2. Change `TURN_SPEED`. What is too fast? What is too slow?
3. Walk the teddy around the robot. Does it keep facing the teddy?

# Session 7: Drive to the teddy
1. Width = distance
2. Calibrate `target_w`
3. Smooth the width
4. Speed tiers
5. Laser safety stop
6. The full program

## 1. Width tells distance
The **closer** the teddy, the **wider** its box. In Session 4 your table showed this.

We pick a stop width, `target_w`: the width of the box when the teddy is 30 cm away.

## 2. Calibrate `target_w`
Place the teddy **30 cm** in front of the robot (measure it!). Run this cell.

In [ ]:
widths = []
for i in range(10):
    frame = get_frame()
    result = find_teddy_best(frame)
    if result is not None:
        cx,w,conf = result
        widths.append(w)
    time.sleep(0.2)

print("widths:",widths)
if widths:
    print(f"average width = {sum(widths)/len(widths):.0f}")

In [ ]:
# type your calibration result here
TARGET_W = 200

## 3. Smooth the width
One box can jump bigger or smaller. We average the last 3 widths.

In [ ]:
recent_widths = []

def smooth_width(w):
    recent_widths.append(w)
    if len(recent_widths) > 3:
        recent_widths.pop(0)
    return sum(recent_widths)/len(recent_widths)

# test
for w in [200,210,150,205]:
    print(w,"->",smooth_width(w))

## 4. Speed tiers (no Kp)
```
far    -> fast
closer -> slow
arrived-> stop
```
Fixed numbers only.

In [ ]:
FAST_SPEED = 20
SLOW_SPEED = 10

def decide_speed(w):
    if w < TARGET_W - 60:
        return FAST_SPEED       # far away
    elif w < TARGET_W:
        return SLOW_SPEED       # getting close
    else:
        return 0                # arrived

for test_w in [80,120,150,190,210]:
    print(test_w,decide_speed(test_w))

## 5. Laser safety stop
The laser sensor is the last safety check. If something is closer than `SAFE_CM`, the robot stops, whatever YOLO says.

In [ ]:
SAFE_CM = 15

def too_close():
    distance = got.read_distance_data(21)
    return distance <= SAFE_CM

print("Laser:",got.read_distance_data(21),"cm")

## 6. The full program: find, face, approach
Order of rules in every step:
1. Laser too close? **stop**
2. No teddy? count the miss, search after 5 misses
3. Teddy not centered? **turn**
4. Teddy centered? **drive forward** with the speed tier
5. Speed is 0? **arrived!**

There is also a **time limit**, so the robot always stops.

In [ ]:
def teddy_find_and_approach(time_limit = 40):
    start = time.time()
    missed = 0
    recent_widths.clear()

    while time.time() - start < time_limit:
        # rule 1: safety
        if too_close():
            print("Laser says STOP.")
            break

        frame = fresh_frame()
        result = find_teddy_best(frame)

        # rule 2: lost
        if result is None:
            missed += 1
            if missed >= 5:
                turn_step("right")
            continue
        missed = 0
        cx,w,conf = result
        w_smooth = smooth_width(w)
        print(f"cx = {cx}  w = {w_smooth:.0f}")

        # rule 3: turn
        direction = decide_turn(cx)
        if direction != "centered":
            turn_step(direction)
            continue

        # rule 4 and 5: drive or arrive
        speed = decide_speed(w_smooth)
        if speed == 0:
            got.mecanum_stop()
            got.show_light_rgb_effect(0,255,0,3)
            got.play_audio_tts("Teddy! I am here!")
            print("Arrived!")
            return True
        got.mecanum_move_xyz(0,speed,0)
        time.sleep(0.3)
        got.mecanum_stop()

    got.mecanum_stop()
    print("Time up or stopped.")
    return False

teddy_find_and_approach()
got.turn_off_lights()

**Practice**
1. Change `TARGET_W` so the robot stops 50 cm away.
2. What happens without `smooth_width`? Try it.
3. Put an object between the robot and the teddy. Which rule stops the robot?

---
*Next (Session 8): wrong-object tests, search mode, and your own upgrade.*